Install Important Modules

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os 
from sklearn.preprocessing import StandardScaler
no_row=300

Create the Dataset


In [3]:
if not os.path.exists('./Data'): os.makedirs('../Data')
dist = np.random.uniform(2, 50, no_row)
data = pd.DataFrame({
    'distance_km': dist,
    'time_of_day': np.random.choice(['Morning', 'Afternoon', 'Evening', 'Night'], no_row),
    'weather': np.random.choice(['Sunny', 'Rainy', 'Snowy'], no_row),
    'ride_type': np.random.choice(['Economy', 'Premium', 'Shared', 'Luxury'], no_row),
    'day_type': np.random.choice(['weekday', 'weekend'], no_row)
})

traffic_map = {'Morning': 2.0, 'Evening': 2.0, 'Afternoon': 1.4, 'Night': 1.0}
weather_map = {'Sunny': 0, 'Rainy': 5, 'Snowy': 10}
data['duration_min'] = (data['distance_km'] * 1.5 * data['time_of_day'].map(traffic_map)) + data['weather'].map(weather_map) + 5

base_p = 5.0 + (data['distance_km'] * 1.2) + (data['duration_min'] * 0.5)
type_mult = {'Luxury': 2.0, 'Premium': 1.5, 'Economy': 1.0, 'Shared': 0.8}
data['ride_price'] = base_p * data['ride_type'].map(type_mult)

# Using a very low noise level to ensure the model can capture the primary signal for high accuracy
data['ride_price'] += np.random.normal(0, 0.01, no_row)

# Adding small data quality issues to test our cleaning code
data.loc[0:10, 'distance_km'] = np.nan
data.loc[20:25, 'ride_price'] = 9999.0

data.to_csv('../Data/rides.csv', index=False)
print(f"High-accuracy sample dataset with {len(data)} rows created.")

High-accuracy sample dataset with 300 rows created.


Data Proccesing

In [8]:
df = pd.read_csv('../data/rides.csv')
df = df.drop_duplicates()

# 1. Fixing missing values by using the median
df['distance_km'] = df['distance_km'].fillna(df['distance_km'].median())

# 2. Fixing outliers using the IQR method
for col in ['distance_km', 'duration_min', 'ride_price']:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    df.loc[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR), col] = df[col].median()

# 3. Encoding text into numbers
df_proc = pd.get_dummies(df, columns=['time_of_day', 'weather', 'ride_type', 'day_type'], drop_first=True)

# 4. High-Accuracy Engineering: Cross-Interaction Terms
# This step is critical because our pricing formula uses multiplication.
#  manually create interaction columns to help the Linear model see these patterns.
categorical_cols = [c for c in df_proc.columns if '_' in c and c != 'ride_price']
for col in categorical_cols:
    df_proc[f'cross_{col}'] = df_proc['distance_km'] * df_proc[col]

# 5. Scaling all numerical features
scaler = StandardScaler()
feat_cols = [c for c in df_proc.columns if c != 'ride_price']
df_proc[feat_cols] = scaler.fit_transform(df_proc[feat_cols])

print("Processing complete. All features have been cleaned and optimized.")

Processing complete. All features have been cleaned and optimized.
